In [ ]:
# Instala as bibliotecas necessárias direto no contêiner do Jupyter
%pip install kafka-python boto3 pillow --quiet

import json
import time
from datetime import datetime
from kafka import KafkaConsumer
import boto3

In [ ]:
# Configura o cliente do MinIO (Simulando o AWS S3)
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='root-minio',
    aws_secret_access_key='root12345678',
    region_name='us-east-1'
)

# Garante que o bucket do datalake existe
BUCKET_NAME = "datalake"
try:
    s3_client.create_bucket(Bucket=BUCKET_NAME)
except Exception:
    pass # Se já existir, segue o jogo

# 2. Configura o Consumidor do Kafka
consumer = KafkaConsumer(
    'cdc.public.pedidos', # Tópico gerado pelo Debezium
    bootstrap_servers=['kafka:29092'],
    auto_offset_reset='earliest', # Começa do início do tópico
    enable_auto_commit=True,      # Confirma a leitura automaticamente
    group_id='grupo-python-consumer',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) # Transforma bytes em dicionário Python
)

print("Conexões estabelecidas! Pronto para consumir...")

In [ ]:
# Configurações do Buffer de Micro-Batch para proteção do Object Storage
buffer_mensagens = []
ULTIMO_FLUSH = time.time()
LIMITE_MENSAGENS = 5             # Gatilho por volumetria
INTERVALO_TEMPO_SEGUNDOS = 10     # Gatilho por tempo (segurança)

print("Pipeline de Streaming Ativo!")
print("Aguardando eventos do Postgres/Debezium... (Pressione o botão 'Stop' para encerrar)")
print("-" * 80)

try:
    while True:
        # Busca novas mensagens do Kafka (espera ativa de até 1 segundo)
        mensagens_recebidas = consumer.poll(timeout_ms=1000)
        
        for tp, msgs in mensagens_recebidas.items():
            for msg in msgs:
                payload_completo = msg.value
                
                # Valida se a mensagem possui a estrutura padrão do Debezium Connect
                if payload_completo and 'payload' in payload_completo:
                    payload = payload_completo['payload']
                    operacao = payload.get('op') # c = insert, u = update, d = delete
                    
                    # --------------------------------------------------------
                    # CASO A: Exclusão Física no Banco (DELETE)
                    # No delete, o dado sumiu, então o Debezium envia o estado antigo no 'before'
                    # --------------------------------------------------------
                    if operacao == 'd': 
                        dados_pedido = payload.get('before')
                        if dados_pedido:
                            dados_pedido['tipo_operacao_cdc'] = 'DELETE'
                            buffer_mensagens.append(dados_pedido)
                            print(f"❌ [DELETE] Registro apagado na origem! ID Pedido: {dados_pedido['id_pedido']}")
                    
                    # --------------------------------------------------------
                    # CASO B: Inclusão ou Alteração no Banco (INSERT / UPDATE)
                    # O dado atualizado ou novo vem mapeado dentro do campo 'after'
                    # --------------------------------------------------------
                    elif operacao in ['c', 'u']: 
                        dados_pedido = payload.get('after')
                        if dados_pedido:
                            dados_pedido['tipo_operacao_cdc'] = 'INSERT' if operacao == 'c' else 'UPDATE'
                            buffer_mensagens.append(dados_pedido)
                            print(f"📥 [EVENTO] Capturado! ID: {dados_pedido['id_pedido']} | Status: {dados_pedido['status']} ({dados_pedido['tipo_operacao_cdc']})")

        # --------------------------------------------------------------------
        # Mecanismo de Flush: Decisão de escrita no Data Lake
        # --------------------------------------------------------------------
        tempo_decorrido = time.time() - ULTIMO_FLUSH
        
        # Dispara a escrita se bater a quantidade limite OU se estourar o tempo com dados na fila
        if len(buffer_mensagens) >= LIMITE_MENSAGENS or (tempo_decorrido >= INTERVALO_TEMPO_SEGUNDOS and len(buffer_mensagens) > 0):
            
            # Define partição temporal para a estrutura de arquivos do Data Lake
            timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
            nome_arquivo = f"live/pedidos/pedidos_batch_{timestamp_str}.jsonl"
            
            # Transforma os dicionários da memória em formato JSON Lines (um JSON por linha)
            conteudo_jsonl = "\n".join([json.dumps(m) for m in buffer_mensagens])
            
            # Persiste o lote diretamente dentro do MinIO
            s3_client.put_object(
                Bucket=BUCKET_NAME,
                Key=nome_arquivo,
                Body=conteudo_jsonl.encode('utf-8')
            )
            
            print(f"[DATA LAKE] Flush realizado. {len(buffer_mensagens)} mensagens salvas em: {nome_arquivo}")
            print("-" * 80)
            
            # Limpa o estado da memória e reseta o temporizador
            buffer_mensagens.clear()
            ULTIMO_FLUSH = time.time()

except KeyboardInterrupt:
    # --------------------------------------------------------------------
    # SALVA-VIDAS (Graceful Shutdown)
    # Se o usuário parar o Jupyter, impede que dados fiquem órfãos na memória
    # --------------------------------------------------------------------
    print("\nSinal de interrupção detectado! Iniciando rotina de desligamento seguro...")
    
    if len(buffer_mensagens) > 0:
        print(f"Atenção: Detectadas {len(buffer_mensagens)} mensagens pendentes no buffer.")
        print("Forçando persistência dos dados remanescentes no MinIO...")
        
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        nome_arquivo = f"live/pedidos/pedidos_final_shutdown_{timestamp_str}.jsonl"
        conteudo_jsonl = "\n".join([json.dumps(m) for m in buffer_mensagens])
        
        s3_client.put_object(
            Bucket=BUCKET_NAME,
            Key=nome_arquivo,
            Body=conteudo_jsonl.encode('utf-8')
        )
        print(f"Backup de segurança concluído! Arquivo salvo: {nome_arquivo}")
    else:
        print("Buffer estava limpo. Nenhum dado em risco de perda.")

finally:
    # Desconecta do cluster de mensageria liberando os recursos do contêiner
    consumer.close()
    print("🔌 Conexão com o broker Kafka encerrada com sucesso. Sistema offline.")